# Train Model 

## Cell 1 – Imports

In [1]:
import json
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EvalPrediction,
    TrainerCallback,
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

from tqdm import tqdm

HEURISTIC_NAMES = [
    "parallelism",
    "task_elimination",
    "task_automation",
    "task_composition",
    "case_based_work",
    "numerical_involvement",
    "knock_out",
]

NUM_LABELS = len(HEURISTIC_NAMES)

## Cell 2 – Load Dataset

In [ ]:
# processed_root = Path(r"C:\Users\yousu\Downloads\SAP\project\data\processed")

# train_dir = processed_root / "train"
# eval_dir = processed_root / "eval"


# def load_records(folder: Path):
#     records = []

#     for file in sorted(folder.glob("*.json")):
#         with open(file, encoding="utf-8") as f:
#             records.append(json.load(f))

#     return records


# final_train_records = load_records(train_dir)
# final_eval_records = load_records(eval_dir)

# print(f"Training Records : {len(final_train_records)}")
# print(f"Evaluation Records : {len(final_eval_records)}")

In [13]:
processed_root = Path(r"C:\Users\yousu\Downloads\SAP\project\data\processed")

train_dir = processed_root / "train"
eval_dir = processed_root / "eval"


def load_records(folder: Path):
    records = []

    for file in sorted(folder.glob("*.json")):
        with open(file, encoding="utf-8") as f:
            records.append(json.load(f))

    return records


final_train_records = load_records(train_dir)
final_eval_records = load_records(eval_dir)

print(f"Total:-")
print(f"Training Records : {len(final_train_records)}")
print(f"Evaluation Records : {len(final_eval_records)}")

MAX_TRAIN = 20000
MAX_EVAL = 4000

final_train_records = final_train_records[:MAX_TRAIN]
final_eval_records = final_eval_records[:MAX_EVAL]

print(f"Sampled:-")
print(f"Training Records : {len(final_train_records)}")
print(f"Evaluation Records : {len(final_eval_records)}")

Total:-
Training Records : 88183
Evaluation Records : 22046
Sampled:-
Training Records : 20000
Evaluation Records : 4000


## Cell 3 – Build Prompt & Labels

In [14]:
def record_to_prompt(record):

    process = record["as-is"]

    tasks = [
        task["task"]["task_name"]
        for task in sorted(
            process["process_task"],
            key=lambda x: x["order"]
        )
    ]

    gateways = [
        f'{g["gateway_type"]} ({g["name"]})'
        for g in process["gateways"]
    ]

    prompt = f"""
### Process Name
{process["process_name"]}

### Tasks
{" -> ".join(tasks)}

### Gateways
{", ".join(gateways) if gateways else "None"}

### Statistics

Total Time : {process["capacity_requirement_minutes"]} minutes

Number of Tasks : {len(tasks)}

Number of Gateways : {len(gateways)}
"""

    return prompt.strip()


def record_to_labels(record):

    applied = {
        item["heuristicName"]: item["isApplied"]
        for item in record["redesignTrace"]
    }

    return [
        1.0 if applied.get(name, False) else 0.0
        for name in HEURISTIC_NAMES
    ]


def build_dataset(records):

    texts = [record_to_prompt(r) for r in records]

    labels = [record_to_labels(r) for r in records]

    return Dataset.from_dict(
        {
            "text": texts,
            "labels": labels,
        }
    )


train_ds_raw = build_dataset(final_train_records)
eval_ds_raw = build_dataset(final_eval_records)

print(train_ds_raw[0])

{'text': '### Process Name\n710 BPMN Conference room\n\n### Tasks\nEntering Room Requirements -> Task 2 -> Show Alternative -> Check Later -> Selecting Room and Time Slot -> Enter Order Requirements -> Inventory and Food Required -> Inventory Confirmed -> Pay Bill -> Display rooms\n\n### Gateways\nEXCLUSIVE (Find fit?), EXCLUSIVE (Find fit?)\n\n### Statistics\n\nTotal Time : 1752 minutes\n\nNumber of Tasks : 10\n\nNumber of Gateways : 2', 'labels': [1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0]}


## Cell 4 – Tokenization

In [15]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"


def tokenize_function(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )


train_ds = train_ds_raw.map(
    tokenize_function,
    batched=True,
)

eval_ds = eval_ds_raw.map(
    tokenize_function,
    batched=True,
)

train_ds.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels",
    ],
)

eval_ds.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels",
    ],
)

print(train_ds)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 20000
})


## Cell 5 – Load Qwen + LoRA

In [16]:
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    trust_remote_code=True,
)

model.config.pad_token_id = tokenizer.pad_token_id

model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

model = get_peft_model(
    model,
    lora_config,
)

model.print_trainable_parameters()

print()

print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU :", torch.cuda.get_device_name(0))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 2,168,960 || all params: 496,208,000 || trainable%: 0.4371

CUDA Available : True
GPU : NVIDIA GeForce RTX 4050 Laptop GPU


## Cell 6 – Metrics

In [17]:
def compute_metrics(eval_pred: EvalPrediction):

    logits, labels = eval_pred

    probabilities = torch.sigmoid(
        torch.tensor(logits)
    ).numpy()

    predictions = (probabilities >= 0.5).astype(int)

    labels = labels.astype(int)

    micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="micro",
        zero_division=0,
    )

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    label_precision, label_recall, label_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average=None,
        zero_division=0,
    )

    subset_accuracy = accuracy_score(
        labels,
        predictions,
    )

    metrics = {
        "subset_accuracy": subset_accuracy,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
    }

    for index, name in enumerate(HEURISTIC_NAMES):

        metrics[f"precision_{name}"] = label_precision[index]
        metrics[f"recall_{name}"] = label_recall[index]
        metrics[f"f1_{name}"] = label_f1[index]

    return metrics

## Cell 7 – Progress Bar

In [18]:
class TqdmProgressCallback(TrainerCallback):

    def on_train_begin(
        self,
        args,
        state,
        control,
        **kwargs,
    ):
        self.progress = tqdm(
            total=state.max_steps,
            desc="Training",
        )

    def on_step_end(
        self,
        args,
        state,
        control,
        **kwargs,
    ):
        self.progress.update(1)

    def on_train_end(
        self,
        args,
        state,
        control,
        **kwargs,
    ):
        self.progress.close()

## Cell 8 – Training Arguments

In [19]:
training_args = TrainingArguments(

    output_dir="./qwen_redesign_finetune",

    num_train_epochs=5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    learning_rate=2e-4,

    weight_decay=0.01,

    warmup_steps=100,

    logging_steps=20,

    eval_strategy="epoch",

    save_strategy="epoch",

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="micro_f1",

    greater_is_better=True,

    bf16=torch.cuda.is_available(),

    fp16=False,

    report_to="none",

    seed=42,
)

## Cell 9 – Trainer

In [20]:
import torch

print(torch.cuda.is_bf16_supported())

True


In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[TqdmProgressCallback()],
)

## Cell 10 – Train Model

In [22]:
print("Model Device:", next(model.parameters()).device)
import torch

print("CUDA Available :", torch.cuda.is_available())
print("CUDA Version   :", torch.version.cuda)
print("GPU Count      :", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU Name       :", torch.cuda.get_device_name(0))

Model Device: cuda:0
CUDA Available : True
CUDA Version   : 12.9
GPU Count      : 1
GPU Name       : NVIDIA GeForce RTX 4050 Laptop GPU


In [23]:
print()

print("Training Started")

print()

train_result = trainer.train()

print()

print("Training Finished")

print()

print(train_result)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



Training Started



Training:   0%|          | 1/12500 [00:00<3:18:04,  1.05it/s]

Epoch,Training Loss,Validation Loss,Subset Accuracy,Micro Precision,Micro Recall,Micro F1,Macro Precision,Macro Recall,Macro F1,Precision Parallelism,Recall Parallelism,F1 Parallelism,Precision Task Elimination,Recall Task Elimination,F1 Task Elimination,Precision Task Automation,Recall Task Automation,F1 Task Automation,Precision Task Composition,Recall Task Composition,F1 Task Composition,Precision Case Based Work,Recall Case Based Work,F1 Case Based Work,Precision Numerical Involvement,Recall Numerical Involvement,F1 Numerical Involvement,Precision Knock Out,Recall Knock Out,F1 Knock Out
1,0.401929,0.236591,0.483000,0.845789,0.881923,0.863478,0.737339,0.594551,0.605222,0.740182,0.957906,0.835086,0.989659,0.864499,0.922854,0.986187,0.996328,0.991231,0.000000,0.000000,0.000000,1.000000,0.226667,0.369565,0.783006,0.957580,0.861538,0.662338,0.158879,0.256281
2,0.394822,0.206249,0.510750,0.861986,0.882129,0.871941,0.724140,0.711358,0.704679,0.842161,0.816222,0.828989,0.922948,0.995483,0.957844,0.986192,0.996695,0.991416,0.000000,0.000000,0.000000,0.898734,0.946667,0.922078,0.776361,0.961197,0.858948,0.642586,0.263240,0.373481
3,0.333730,0.191614,0.539250,0.869783,0.892526,0.881008,0.733936,0.729293,0.727287,0.839488,0.875257,0.856999,0.989973,0.981030,0.985481,0.989059,0.995960,0.992498,0.000000,0.000000,0.000000,0.933333,0.933333,0.933333,0.789590,0.937849,0.857358,0.596107,0.381620,0.465337
4,0.240310,0.206398,0.530500,0.872737,0.883158,0.877916,0.733153,0.721712,0.725546,0.848756,0.875770,0.862052,0.977698,0.990063,0.983842,0.993030,0.994124,0.993577,0.000000,0.000000,0.000000,0.941176,0.853333,0.895105,0.799119,0.894771,0.844244,0.572289,0.443925,0.500000
5,0.241620,0.241007,0.508250,0.873678,0.867202,0.870428,0.740334,0.715865,0.723020,0.856559,0.861396,0.858971,0.978456,0.984643,0.981540,0.991587,0.995593,0.993586,0.062500,0.005618,0.010309,0.943662,0.893333,0.917808,0.799572,0.859257,0.828340,0.550000,0.411215,0.470588


Training: 100%|██████████| 12500/12500 [2:26:53<00:00,  1.42it/s]   



Training Finished

TrainOutput(global_step=12500, training_loss=0.36655430448532106, metrics={'train_runtime': 8813.8444, 'train_samples_per_second': 11.346, 'train_steps_per_second': 1.418, 'total_flos': 5.53072656384e+16, 'train_loss': 0.36655430448532106, 'epoch': 5.0})


## Cell 11 – Evaluation & Save Model

In [24]:
print()

print("Running Evaluation")

print()

evaluation_metrics = trainer.evaluate()

print()

for key, value in evaluation_metrics.items():
    print(f"{key:30} : {value}")


Running Evaluation



Training Loss,Validation Loss,Epoch,Subset Accuracy,Micro Precision,Micro Recall,Micro F1,Macro Precision,Macro Recall,Macro F1,Precision Parallelism,Recall Parallelism,F1 Parallelism,Precision Task Elimination,Recall Task Elimination,F1 Task Elimination,Precision Task Automation,Recall Task Automation,F1 Task Automation,Precision Task Composition,Recall Task Composition,F1 Task Composition,Precision Case Based Work,Recall Case Based Work,F1 Case Based Work,Precision Numerical Involvement,Recall Numerical Involvement,F1 Numerical Involvement,Precision Knock Out,Recall Knock Out,F1 Knock Out
0.241620,0.191614,5,0.539250,0.869783,0.892526,0.881008,0.733936,0.729293,0.727287,0.839488,0.875257,0.856999,0.989973,0.981030,0.985481,0.989059,0.995960,0.992498,0.000000,0.000000,0.000000,0.933333,0.933333,0.933333,0.789590,0.937849,0.857358,0.596107,0.381620,0.465337



eval_loss                      : 0.19161413609981537
eval_subset_accuracy           : 0.53925
eval_micro_precision           : 0.8697833065810594
eval_micro_recall              : 0.8925262507720816
eval_micro_f1                  : 0.8810080276394675
eval_macro_precision           : 0.7339357592328213
eval_macro_recall              : 0.7292927834782242
eval_macro_f1                  : 0.7272865644114347
eval_precision_parallelism     : 0.8394879369768586
eval_recall_parallelism        : 0.8752566735112937
eval_f1_parallelism            : 0.8569992460417191
eval_precision_task_elimination : 0.9899726526891522
eval_recall_task_elimination   : 0.981029810298103
eval_f1_task_elimination       : 0.985480943738657
eval_precision_task_automation : 0.9890590809628009
eval_recall_task_automation    : 0.9959603378626515
eval_f1_task_automation        : 0.9924977127172918
eval_precision_task_composition : 0.0
eval_recall_task_composition   : 0.0
eval_f1_task_composition       : 0.0
eval_precision

## Cell 12 - Save Model

In [25]:
SAVE_DIRECTORY = "./trained_qwen_classifier"

trainer.save_model(SAVE_DIRECTORY)

tokenizer.save_pretrained(SAVE_DIRECTORY)

print()

print("Model Saved Successfully")

print()

print(SAVE_DIRECTORY)


Model Saved Successfully

./trained_qwen_classifier
